In [1]:
pip install torch torchvision timm pandas pillow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import argparse, os, gc, sys

# Bersihkan sisa model/memori dari run sebelumnya (kalau dijalankan ulang di Jupyter)
for _n in ("model", "opt", "sched", "scaler", "x", "y", "loss"):
    globals().pop(_n, None)
if hasattr(sys, "last_traceback"):
    sys.last_traceback = None
    sys.last_value = None
gc.collect()

# ====== UBAH DI SINI SESUAI LAPTOPMU ======
DATA_DIR = "/Users/rikata/Documents/SMS5/SupervisedML/kaggle_Comp/Document-Image-Classification"                          
OUT_PATH = "/Users/rikata/Documents/SMS5/SupervisedML/kaggle_Comp/Document-Image-Classification/submission/submission.csv"  
# ==========================================
import pandas as pd, numpy as np, torch, timm
from PIL import Image, ImageFile
import warnings
warnings.filterwarnings("ignore", message="Corrupt EXIF data")
ImageFile.LOAD_TRUNCATED_IMAGES = True 
BAD_FILES = set()
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

p = argparse.ArgumentParser()
p.add_argument("--data_dir", default=DATA_DIR)
p.add_argument("--epochs", type=int, default=3)
p.add_argument("--bs", type=int, default=8)
p.add_argument("--size", type=int, default=224)
p.add_argument("--model", default="convnext_tiny.fb_in22k_ft_in1k")
p.add_argument("--out", default=OUT_PATH)
a, _ = p.parse_known_args() 

dev = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
if dev == "mps":
    torch.mps.empty_cache()
print("device:", dev)
print(f"CONFIG -> size={a.size} bs={a.bs} epochs={a.epochs} model={a.model}")
use_amp = dev == "cuda"
norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# Tanpa flip horizontal: tata letak dokumen punya arah baca yang bermakna.
tf_train = T.Compose([T.Resize((a.size, a.size)), T.RandomAffine(3, translate=(0.02, 0.02), scale=(0.95, 1.05), fill=255),
                      T.ToTensor(), norm])
tf_test = T.Compose([T.Resize((a.size, a.size)), T.ToTensor(), norm])


def resolve(fn):
    """Cocokkan path CSV (train/resume/x.tif) dengan folder bersarang (train/train/resume/x.tif)."""
    top, rest = fn.split("/", 1)
    for c in (fn, f"{top}/{top}/{rest}"):
        pth = os.path.join(a.data_dir, c)
        if os.path.exists(pth):
            return pth
    raise FileNotFoundError(f"{fn} tidak ditemukan di {a.data_dir}")


class DS(Dataset):
    def __init__(self, df, tf, labels=True):
        self.df, self.tf, self.labels = df.reset_index(drop=True), tf, labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        try:
            img = Image.open(resolve(r.filename)).convert("RGB")
        except Exception as e:
            if r.filename not in BAD_FILES:
                BAD_FILES.add(r.filename)
                print(f"[WARN] gagal baca {r.filename}: {type(e).__name__}")
            img = Image.new("RGB", (a.size, a.size), (255, 255, 255))
        x = self.tf(img)
        return (x, int(r.label)) if self.labels else x


tr = pd.read_csv(os.path.join(a.data_dir, "train.csv"))
te = pd.read_csv(os.path.join(a.data_dir, "test.csv"))
resolve(tr.filename[0]); resolve(te.filename[0])  
val = tr.sample(frac=0.05, random_state=0)
tr = tr.drop(val.index)

dl_tr = DataLoader(DS(tr, tf_train), a.bs, shuffle=True, num_workers=0 if dev == "mps" else 4, drop_last=True)
dl_va = DataLoader(DS(val, tf_test), a.bs * 2, num_workers=0 if dev == "mps" else 4)
dl_te = DataLoader(DS(te, tf_test, labels=False), a.bs * 2, num_workers=0 if dev == "mps" else 4)

model = timm.create_model(a.model, pretrained=True, num_classes=16).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-4, total_steps=a.epochs * len(dl_tr))
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
crit = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

for ep in range(a.epochs):
    model.train()
    for x, y in dl_tr:
        x, y = x.to(dev), y.to(dev)
        with torch.autocast("cuda", enabled=use_amp):
            loss = crit(model(x), y)
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        if dev == "mps":
            torch.mps.empty_cache()
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in dl_va:
            correct += (model(x.to(dev)).argmax(1).cpu() == y).sum().item()
    print(f"epoch {ep+1}/{a.epochs}  val_acc={correct/len(val):.4f}", flush=True)

model.eval()
preds = []
with torch.no_grad():
    for x in dl_te:
        preds.append(model(x.to(dev)).argmax(1).cpu().numpy())
preds = np.concatenate(preds)

pred_map = dict(zip(te["id"], preds))
sample = pd.read_csv(os.path.join(a.data_dir, "sample_submission.csv"))
submission = sample.copy()
submission["label"] = submission["id"].map(pred_map).astype(int)
assert submission["label"].between(0, 15).all() and len(submission) == len(sample)
os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
submission.to_csv(a.out, index=False)
sub = submission
print("Saved", a.out, sub.shape)

/Users/rikata/Documents/SMS5/SupervisedML/kaggle_Comp/document-image-classification/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps
CONFIG -> size=224 bs=8 epochs=3 model=convnext_tiny.fb_in22k_ft_in1k
[WARN] gagal baca train/scientific_publication/2500126531_2500126536.tif: UnidentifiedImageError
epoch 1/3  val_acc=0.7688
epoch 2/3  val_acc=0.8400
epoch 3/3  val_acc=0.8619
Saved /Users/rikata/Documents/SMS5/SupervisedML/kaggle_Comp/Document-Image-Classification/submission/submission.csv (7994, 2)
